# D102 — File Generator Example

This notebook uses the MovieLens `movies.csv` dataset to demonstrate generators over directories and files.

## What we will do

1. Locate the MovieLens source file.
2. Split it into ten smaller CSV files under `movie-parts`.
3. Walk through the part files in a predictable order.
4. Open one file at a time and yield one line at a time.
5. Consume lines using `next()`, a `for` loop, and `itertools.islice()`.
6. Optionally split a line into simple fields without doing full CSV analysis.

The focus is lazy file processing, not MovieLens data analysis.

## 1. Paths used in this example

The source dataset in this environment is located at:

`C:\\data\\movielens\\ml-latest-small\\movies\\movies.csv`

The ten generated files are placed in:

`C:\\data\\movielens\\ml-latest-small\\movie-parts`

`pathlib.Path` keeps the path operations readable and works well with directory iteration.

In [ ]:
from pathlib import Path

MOVIELENS_DIR = Path(r"C:\data\movielens\ml-latest-small")
SOURCE_FILE = MOVIELENS_DIR / "movies" / "movies.csv"
PARTS_DIR = MOVIELENS_DIR / "movie-parts"

print("Source:", SOURCE_FILE)
print("Source exists:", SOURCE_FILE.is_file())
print("Parts directory:", PARTS_DIR)

## 2. Split `movies.csv` into ten files

For this small demonstration dataset, we read the lines into memory once and divide the data rows as evenly as possible. Every part receives the original header, making each part a valid standalone CSV file.

The later reading example is different: it uses a generator and does **not** load all ten parts into memory.

Running this cell again safely replaces the ten named part files.

In [ ]:
def split_movie_file(source_file, output_dir, number_of_parts=10):
    """Split a small CSV file evenly and repeat its header in every part."""
    if number_of_parts <= 0:
        raise ValueError("number_of_parts must be greater than zero")
    if not source_file.is_file():
        raise FileNotFoundError(source_file)

    output_dir.mkdir(parents=True, exist_ok=True)

    with source_file.open("r", encoding="utf-8") as source:
        header = source.readline()
        movie_lines = source.readlines()

    base_size, extra_rows = divmod(len(movie_lines), number_of_parts)
    offset = 0
    created_files = []

    for part_number in range(1, number_of_parts + 1):
        row_count = base_size + (1 if part_number <= extra_rows else 0)
        part_file = output_dir / f"movies-part-{part_number:02}.csv"

        with part_file.open("w", encoding="utf-8", newline="") as output:
            output.write(header)
            output.writelines(movie_lines[offset:offset + row_count])

        print(f"Created {part_file.name}: {row_count} movie rows")
        created_files.append(part_file)
        offset += row_count

    return created_files

In [ ]:
part_files = split_movie_file(SOURCE_FILE, PARTS_DIR, number_of_parts=10)
print("\nTotal part files:", len(part_files))

## 3. Walk through the directory

`Path.glob()` returns an iterable directory search result. We sort the paths so the lesson always processes part 01 through part 10 in order.

Only files matching `movies-part-*.csv` are selected. Unrelated files or subdirectories are ignored.

In [ ]:
def movie_part_paths(folder):
    """Yield matching movie-part paths in filename order."""
    print(f"PATH GENERATOR: scanning {folder}")

    for path in sorted(folder.glob("movies-part-*.csv")):
        if path.is_file():
            print(f"PATH GENERATOR: found {path.name}")
            yield path

In [ ]:
path_generator = movie_part_paths(PARTS_DIR)
print("Created path generator; directory has not been scanned yet.\n")

print("First path requested with next():", next(path_generator).name)
print("Second path requested with next():", next(path_generator).name)

path_generator.close()

The directory scan starts only at the first `next()` request. Each request resumes the generator after its previous `yield`.

## 4. Yield lines across all files

The next generator combines two levels of iteration:

- Outer loop: one file path at a time.
- Inner loop: one line from the open file at a time.

A file object is itself an iterator over lines. Therefore, `for line in movie_file` reads lazily rather than calling `readlines()`.

The generator yields `(file_name, line_number, line)` so the consumer can see where every line originated.

In [ ]:
def movie_lines(folder, skip_repeated_headers=True):
    """Yield lines lazily from every MovieLens part file."""
    first_file = True

    for file_path in movie_part_paths(folder):
        print(f"FILE GENERATOR: opening {file_path.name}")

        with file_path.open("r", encoding="utf-8") as movie_file:
            for line_number, line in enumerate(movie_file, start=1):
                if skip_repeated_headers and not first_file and line_number == 1:
                    print(f"FILE GENERATOR: skipping repeated header in {file_path.name}")
                    continue

                clean_line = line.rstrip("\r\n")
                print(
                    f"FILE GENERATOR: yielding {file_path.name}, "
                    f"line {line_number}"
                )
                yield file_path.name, line_number, clean_line

        print(f"FILE GENERATOR: closed {file_path.name}")
        first_file = False

### Consumer-side use with `next()`

Creating the generator does not scan or open anything. The first call to `next()` opens the first file and returns its first line. The generator then pauses while the file remains at its current position.

In [ ]:
lines = movie_lines(PARTS_DIR)
print("CONSUMER: generator created\n")

for request_number in range(1, 6):
    file_name, line_number, line = next(lines)
    print(f"CONSUMER request {request_number}: {file_name}:{line_number} -> {line}")

# We intentionally stop early, so explicitly close the generator.
lines.close()

Calling `.close()` is useful when manually stopping a generator that currently owns an open file. It terminates the generator, exits the `with` block, and releases the file handle. A fully consumed `for` loop closes each file naturally.

## 5. Consume only a small sample with `islice()`

`itertools.islice(iterator, n)` lazily requests at most `n` items. It is useful when previewing a large file stream without loading or printing everything.

In [ ]:
from itertools import islice

sample_lines = movie_lines(PARTS_DIR)

try:
    for file_name, line_number, line in islice(sample_lines, 8):
        print(f"SAMPLE {file_name}:{line_number} -> {line}")
finally:
    sample_lines.close()

## 6. Crossing from one file to the next

To make the file boundary easy to see without printing hundreds of lines, the following generator yields only the first two data rows from each file. It still opens files one at a time and reads each selected line lazily.

In [ ]:
def first_movie_lines_per_file(folder, rows_per_file=2):
    for file_path in movie_part_paths(folder):
        print(f"OPEN: {file_path.name}")

        with file_path.open("r", encoding="utf-8") as movie_file:
            next(movie_file, None)  # consume this part's header

            for line_number, line in enumerate(
                islice(movie_file, rows_per_file), start=2
            ):
                yield file_path.name, line_number, line.rstrip("\r\n")

        print(f"CLOSE: {file_path.name}")

In [ ]:
for file_name, line_number, line in first_movie_lines_per_file(PARTS_DIR):
    print(f"CONSUMER: {file_name}:{line_number} -> {line}")

## 7. Optional simple parsing

Movie titles may contain commas inside quoted text, so `line.split(',')` is not a correct general CSV parser. For a lightweight demonstration, `split(',', maxsplit=1)` safely separates only the numeric `movieId` from the remainder of each MovieLens line.

Use Python's `csv` module when all CSV fields must be parsed correctly.

In [ ]:
def simple_movie_records(line_source):
    """Convert each non-header line to a minimal dictionary."""
    for file_name, line_number, line in line_source:
        if line.startswith("movieId,"):
            continue

        movie_id, remaining_text = line.split(",", maxsplit=1)
        yield {
            "source_file": file_name,
            "line_number": line_number,
            "movie_id": int(movie_id),
            "remaining_text": remaining_text,
        }

In [ ]:
line_source = movie_lines(PARTS_DIR)
records = simple_movie_records(line_source)

try:
    for record in islice(records, 5):
        print(record)
finally:
    records.close()
    line_source.close()

## 8. Verify the split without loading file contents

The generator below counts lines one at a time. It keeps only an integer counter in memory. Because every part includes a header, the movie-row count is `line_count - 1`.

In [ ]:
def part_statistics(folder):
    for file_path in movie_part_paths(folder):
        with file_path.open("r", encoding="utf-8") as movie_file:
            line_count = sum(1 for _ in movie_file)

        yield {
            "file": file_path.name,
            "movie_rows": line_count - 1,
            "bytes": file_path.stat().st_size,
        }

total_movie_rows = 0

for statistics in part_statistics(PARTS_DIR):
    total_movie_rows += statistics["movie_rows"]
    print(statistics)

print("Total movie rows across all parts:", total_movie_rows)

## 9. What happens behind the generator?

For `movie_lines()`:

1. Calling `movie_lines(PARTS_DIR)` creates a generator; no directory or file is read yet.
2. The consumer calls `next()` or starts a `for` loop.
3. The generator finds the next matching path and opens that file.
4. The file iterator reads the next line.
5. `yield` returns the tuple to the consumer and suspends the generator.
6. Local variables, loop positions, and the current file position are preserved.
7. The next consumer request resumes immediately after `yield`.
8. At the end of a file, its `with` block closes it before the next file is opened.
9. After the final file, the generator finishes and signals `StopIteration`.

This all happens synchronously on the consumer's thread. The generator is not a background reader.

## 10. Why this approach helps

- Only one file needs to be open at a time.
- Only one current line is required in memory.
- The consumer can stop early without reading all ten files.
- The same consumer loop can work with ten files or thousands of files.
- File discovery, file reading, optional transformation, and consumption stay separate.

For real production CSV work, prefer `csv.reader`, `pandas.read_csv(..., chunksize=...)`, or another parser that correctly handles quoting and data types. The simple string handling here is intentionally limited to the generator lesson.

## 11. Summary

- A directory iterator can lazily provide paths.
- An open text file is an iterator that lazily provides lines.
- A generator can combine both levels and expose one continuous line stream.
- `next()` requests exactly one item; `for` repeatedly requests items until `StopIteration`.
- `islice()` is useful for taking a small lazy sample.
- `with` closes each file reliably when its work is complete.
- If manual consumption stops while the generator owns a resource, call `.close()` or use a `try/finally` cleanup pattern.

## 12. Practice

1. Change the sample from 8 lines to 3 lines.
2. Modify `movie_part_paths()` to yield files in reverse filename order.
3. Write a generator that yields only lines containing `Comedy`.
4. Count how many lines are yielded before the first `Drama` line appears.
5. Replace the simple split with `csv.reader` and yield `movieId`, `title`, and `genres`.